# 🛡️ NetGuard - Cyber Risk Assessment & Threat Intelligence Platform

**Run each cell in order from top to bottom.**

Each cell writes one file to disk using `%%writefile`.

---

### Project Structure
```
netguard/
├── main.ipynb                    ← You are here
├── modules/
│   ├── __init__.py
│   ├── scanner.py                ← Nmap + VirusTotal
│   ├── analyser.py               ← Risk scoring
│   ├── database.py               ← SQLite persistence
│   └── emailer.py                ← Email alert + PDF
├── dashboard/
│   ├── app.py                    ← Main Streamlit page (Overview)
│   └── pages/
│       ├── 2_Scan_Data.py
│       ├── 3_Charts.py
│       ├── 4_Threat_Intel.py
│       └── 5_History.py
├── .env                          ← API keys + email (fill in!)
├── .gitignore
├── requirements.txt
├── license.txt
└── README.md
```

### Recommended Test Targets
```
testphp.vulnweb.com
testasp.vulnweb.com
testaspnet.vulnweb.com
zero.webappsecurity.com
pentest-ground.com
demo.testfire.net
demo.owasp-juice.shop
scanme.nmap.org
```
> ⚠️ Only scan targets you own or have written permission to scan.

In [1]:
# ============================================================
# CELL 1 - Install packages & create folder structure
# ============================================================
import subprocess, sys, os

pkgs = [
    'streamlit', 'plotly', 'requests',
    'python-dotenv', 'fpdf2', 'pandas', 'numpy',
]
for p in pkgs:
    subprocess.run([sys.executable, '-m', 'pip', 'install', p, '-q'], check=True)
    print(f'✅ {p}')

for d in ['modules', 'dashboard/pages', 'scan_results', 'reports']:
    os.makedirs(d, exist_ok=True)

for f in ['modules/__init__.py', 'dashboard/__init__.py']:
    open(f, 'w').close()

print('\n✅ All packages installed and folders created.')
print('▶  Next: run the remaining cells to write each module file.')

✅ streamlit
✅ plotly
✅ requests
✅ python-dotenv
✅ fpdf2
✅ pandas
✅ numpy

✅ All packages installed and folders created.
▶  Next: run the remaining cells to write each module file.


In [2]:
%%writefile .gitignore
.env
__pycache__/
*.pyc
scan_results/
reports/
*.db
.DS_Store
*.egg-info/
dist/
build/

Overwriting .gitignore


In [3]:
%%writefile requirements.txt
streamlit>=1.30.0
plotly>=5.18.0
requests>=2.31.0
python-dotenv>=1.0.0
fpdf2>=2.7.0
pandas>=2.0.0
numpy>=1.26.0

Overwriting requirements.txt


In [4]:
%%writefile license.txt
MIT License

Copyright (c) 2026 Vidzai Digital

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN
THE SOFTWARE.

Overwriting license.txt


In [5]:
%%writefile modules/scanner.py
# modules/scanner.py
# ------------------------------------------------------------------
# Vulnerability Scanning Engine
#   - Nmap port/service detection (65+ ports)
#   - VirusTotal threat intelligence enrichment
#   - CVSS-inspired risk scoring
# ------------------------------------------------------------------
import subprocess
import xml.etree.ElementTree as ET
import time
import os
import re
import requests
import logging

logger = logging.getLogger(__name__)

# Ports to scan - covers most common attack surfaces
SCAN_PORTS = (
    "21,22,23,25,53,67,69,79,80,88,110,111,119,123,135,"
    "137,138,139,143,161,194,389,443,445,500,514,515,587,"
    "631,636,873,993,995,1080,1194,1433,1521,1723,2049,"
    "2375,2376,3306,3389,4444,5432,5900,5985,5986,6379,"
    "6443,6667,7001,8080,8443,8888,9042,9200,9300,9418,"
    "11211,27017,27018,50070,6000,548"
)

SCAN_DIR = "scan_results"
os.makedirs(SCAN_DIR, exist_ok=True)

# ------------------------------------------------------------------
# Vulnerability database
# Each service has: bonus score, display name, CVE ref, CVSS, action
# ------------------------------------------------------------------
VULN_DB = {
    "ftp":           {"bonus":3, "name":"Cleartext FTP",               "cve":"CVE-1999-0497",    "cvss":7.5,  "action":"Disable FTP; use SFTP or FTPS."},
    "telnet":        {"bonus":4, "name":"Cleartext Telnet",            "cve":"CVE-1999-0619",    "cvss":9.8,  "action":"Disable Telnet immediately; replace with SSH."},
    "ssh":           {"bonus":1, "name":"SSH Exposed",                 "cve":"CVE-2023-38408",   "cvss":5.0,  "action":"Restrict SSH; enforce key-based auth only."},
    "smtp":          {"bonus":2, "name":"Open SMTP Relay",             "cve":"CVE-2020-7247",    "cvss":6.5,  "action":"Disable open relay; require SMTP authentication."},
    "http":          {"bonus":1, "name":"Unencrypted HTTP",            "cve":"CWE-319",          "cvss":5.3,  "action":"Enforce HTTPS; install a valid TLS certificate."},
    "https":         {"bonus":0, "name":"HTTPS",                      "cve":"N/A",              "cvss":0.0,  "action":"Ensure TLS certificate is valid and not expired."},
    "http-proxy":    {"bonus":2, "name":"Open HTTP Proxy",             "cve":"CWE-441",          "cvss":6.1,  "action":"Restrict proxy to authorised users only."},
    "http-alt":      {"bonus":1, "name":"Alternate HTTP Port",         "cve":"CWE-16",           "cvss":5.0,  "action":"Restrict access to this alternate HTTP port."},
    "rdp":           {"bonus":4, "name":"RDP Exposed (BlueKeep risk)", "cve":"CVE-2019-0708",    "cvss":9.8,  "action":"Disable RDP or place behind VPN; apply BlueKeep patch."},
    "vnc":           {"bonus":3, "name":"VNC Exposed",                 "cve":"CVE-2006-2369",    "cvss":7.5,  "action":"Restrict VNC; enforce strong passwords and firewall."},
    "mysql":         {"bonus":3, "name":"MySQL Exposed",               "cve":"CVE-2012-2122",    "cvss":7.5,  "action":"Restrict MySQL to localhost; never expose port 3306."},
    "ms-sql-s":      {"bonus":3, "name":"MSSQL Server Exposed",        "cve":"CVE-2020-0618",    "cvss":7.2,  "action":"Firewall port 1433; apply latest SQL Server patches."},
    "smb":           {"bonus":4, "name":"SMB Exposed (EternalBlue)",   "cve":"CVE-2017-0144",    "cvss":9.3,  "action":"Block ports 139/445 at perimeter; apply MS17-010 patch."},
    "netbios-ssn":   {"bonus":3, "name":"NetBIOS Session Exposed",     "cve":"CVE-2017-0144",    "cvss":8.1,  "action":"Block all NetBIOS ports at the firewall."},
    "pop3":          {"bonus":2, "name":"Cleartext POP3",              "cve":"CWE-523",          "cvss":5.9,  "action":"Use POP3S on port 995 with TLS enabled."},
    "imap":          {"bonus":2, "name":"Cleartext IMAP",              "cve":"CWE-523",          "cvss":5.9,  "action":"Use IMAPS on port 993 with TLS enabled."},
    "dns":           {"bonus":2, "name":"Open DNS Resolver",           "cve":"CVE-2008-1447",    "cvss":6.8,  "action":"Disable recursive DNS queries; enable rate limiting."},
    "ntp":           {"bonus":1, "name":"NTP Amplification Risk",      "cve":"CVE-2013-5211",    "cvss":5.0,  "action":"Disable NTP monlist; restrict NTP queries by ACL."},
    "snmp":          {"bonus":3, "name":"SNMP Exposed",                "cve":"CVE-2002-0013",    "cvss":7.8,  "action":"Disable SNMPv1/v2c; upgrade to SNMPv3 with auth."},
    "ldap":          {"bonus":2, "name":"LDAP Exposed",                "cve":"CVE-2021-44228",   "cvss":10.0, "action":"Restrict LDAP access; patch Log4Shell vulnerabilities."},
    "mongodb":       {"bonus":3, "name":"MongoDB No Auth",             "cve":"CVE-2013-2916",    "cvss":7.5,  "action":"Enable MongoDB authentication; bind to localhost."},
    "redis":         {"bonus":4, "name":"Redis Exposed (No Auth)",     "cve":"CVE-2015-4335",    "cvss":9.3,  "action":"Set Redis requirepass; bind to 127.0.0.1 only."},
    "elasticsearch": {"bonus":3, "name":"Elasticsearch Exposed",       "cve":"CVE-2015-1427",    "cvss":10.0, "action":"Enable X-Pack security; firewall port 9200."},
    "postgresql":    {"bonus":2, "name":"PostgreSQL Exposed",          "cve":"CVE-2019-10164",   "cvss":8.8,  "action":"Restrict PostgreSQL to app server only."},
    "memcached":     {"bonus":3, "name":"Memcached Amplification",     "cve":"CVE-2018-1000115", "cvss":7.5,  "action":"Disable Memcached UDP; bind to localhost."},
    "rpcbind":       {"bonus":2, "name":"RPC Portmapper Exposed",      "cve":"CVE-2017-8779",    "cvss":7.5,  "action":"Block port 111 at firewall; disable rpcbind."},
    "nfs":           {"bonus":3, "name":"NFS Share Exposed",           "cve":"CVE-2019-3010",    "cvss":7.8,  "action":"Restrict NFS exports; require strong authentication."},
    "msrpc":         {"bonus":2, "name":"MS-RPC Exposed",              "cve":"CVE-2003-0352",    "cvss":7.5,  "action":"Firewall MS-RPC ports 135 and 445."},
    "irc":           {"bonus":2, "name":"IRC Service Exposed",         "cve":"CVE-2010-2956",    "cvss":6.4,  "action":"Disable IRC; block port 6667 at firewall."},
    "docker":        {"bonus":4, "name":"Docker API Exposed",          "cve":"CVE-2019-5736",    "cvss":9.0,  "action":"Never expose Docker daemon to internet; use TLS."},
    "jenkins":       {"bonus":3, "name":"Jenkins CI Exposed",          "cve":"CVE-2019-1003000", "cvss":9.8,  "action":"Disable anonymous access; place Jenkins behind VPN."},
    "winrm":         {"bonus":3, "name":"WinRM Exposed",               "cve":"CVE-2021-31166",   "cvss":9.8,  "action":"Restrict WinRM; use HTTPS + IP whitelisting."},
    "rsync":         {"bonus":2, "name":"Rsync Exposed",               "cve":"CVE-2014-9512",    "cvss":6.4,  "action":"Restrict rsync with authentication + IP whitelist."},
    "cups":          {"bonus":2, "name":"CUPS Print Server",           "cve":"CVE-2024-47176",   "cvss":9.9,  "action":"Disable CUPS internet exposure immediately."},
    "x11":           {"bonus":3, "name":"X11 Display Server Exposed",  "cve":"CVE-2011-4613",    "cvss":7.5,  "action":"Never expose X11 to internet; use SSH X forwarding."},
    "oracle":        {"bonus":3, "name":"Oracle DB Exposed",           "cve":"CVE-2020-14871",   "cvss":9.8,  "action":"Firewall Oracle DB ports; apply all patches."},
    "unknown":       {"bonus":1, "name":"Unknown Service",             "cve":"N/A",              "cvss":5.0,  "action":"Investigate this service and close if not required."},
}


def get_vuln(service: str) -> dict:
    """Return the vulnerability entry for a service, defaulting to 'unknown'."""
    return VULN_DB.get(service.lower().strip(), VULN_DB["unknown"])


def is_valid_target(target: str) -> bool:
    """Check whether a target is a valid IP, hostname, or CIDR notation."""
    t = target.strip()
    if re.match(r"^\d{1,3}(\.\d{1,3}){3}/\d{1,2}$", t):
        return True
    if re.match(r"^\d{1,3}(\.\d{1,3}){3}$", t):
        return all(0 <= int(p) <= 255 for p in t.split("."))
    if re.match(r"^[a-zA-Z0-9]([a-zA-Z0-9\-]{0,61}[a-zA-Z0-9])?(\.[a-zA-Z]{2,})+$", t):
        return True
    return False


def calc_risk(service: str, malicious_reports: int) -> int:
    """Calculate a risk score 1-10: base 1 + service bonus + VT malicious count."""
    bonus = get_vuln(service)["bonus"]
    return min(10, 1 + bonus + malicious_reports)


def classify_severity(score: int) -> str:
    """Map a numeric risk score to a severity label."""
    if   score >= 9: return "Critical"
    elif score >= 7: return "High"
    elif score >= 4: return "Medium"
    elif score >= 1: return "Low"
    else:            return "Informational"


def run_nmap(target: str) -> str:
    """
    Run an Nmap scan on the given target and save the XML output.
    Returns the path to the saved XML file.

    Flags:
      -Pn    skip host-discovery ping (many hosts block ICMP)
      -sV    detect service names and versions
      --open only show ports that are open
      -p     our custom port list
      -oX    save XML output for Python to parse
    """
    safe   = re.sub(r"[/: ]", "_", target)
    xmlout = os.path.join(SCAN_DIR, f"{safe}.xml")
    cmd = ["nmap", "-Pn", "-sV", "--open", "-p", SCAN_PORTS, "-oX", xmlout, target]
    try:
        subprocess.run(cmd, capture_output=True, timeout=200)
        logger.info(f"Nmap completed for {target}")
    except subprocess.TimeoutExpired:
        logger.warning(f"Nmap timed out for {target}")
    except FileNotFoundError:
        logger.error("Nmap not found - install from https://nmap.org/download.html")
    return xmlout


def parse_nmap_xml(xmlpath: str) -> list:
    """
    Parse the Nmap XML output file.
    Returns a list of dicts, one per open port found.
    """
    rows = []
    if not os.path.exists(xmlpath):
        return rows
    try:
        root = ET.parse(xmlpath).getroot()
        for host in root.findall("host"):
            addr_el = host.find("address")
            if addr_el is None:
                continue
            ip = addr_el.get("addr", "unknown")
            hn_el    = host.find(".//hostname")
            hostname = hn_el.get("name", ip) if hn_el is not None else ip

            for port_el in host.findall(".//port"):
                portid  = port_el.get("portid", "0")
                svc_el  = port_el.find("service")
                svc     = svc_el.get("name",    "unknown") if svc_el is not None else "unknown"
                product = svc_el.get("product", "")        if svc_el is not None else ""
                version = svc_el.get("version", "")        if svc_el is not None else ""
                rows.append({
                    "ip": ip, "hostname": hostname,
                    "port": portid, "service": svc,
                    "product": product, "version": version,
                })
    except Exception as exc:
        logger.error(f"XML parse error for {xmlpath}: {exc}")
    return rows


def check_virustotal(ip: str, api_key: str, retries: int = 3) -> dict:
    """
    Query the VirusTotal v3 API for threat intelligence on an IP address.
    Returns a dict of malicious/suspicious/harmless counts and location info.
    The free tier allows 4 requests per minute - caller should pace calls.
    """
    default = {
        "malicious_reports": 0, "suspicious_count": 0,
        "harmless_count":    0, "community_score":  0,
        "country": "Unknown", "network": "Unknown", "categories": "",
    }
    if not api_key or api_key.startswith("your_"):
        return default

    url     = f"https://www.virustotal.com/api/v3/ip_addresses/{ip}"
    headers = {"x-apikey": api_key}

    for attempt in range(retries):
        try:
            r = requests.get(url, headers=headers, timeout=15)
            if r.status_code == 429:          # rate limit
                wait = 2 ** attempt * 15
                logger.warning(f"VT rate limit for {ip}; waiting {wait}s")
                time.sleep(wait)
                continue
            if r.status_code != 200:
                return default
            attrs = r.json()["data"]["attributes"]
            stats = attrs.get("last_analysis_stats", {})
            cats  = attrs.get("categories", {})
            return {
                "malicious_reports": stats.get("malicious",  0),
                "suspicious_count":  stats.get("suspicious", 0),
                "harmless_count":    stats.get("harmless",   0),
                "community_score":   attrs.get("reputation", 0),
                "country":           attrs.get("country",    "Unknown"),
                "network":           attrs.get("network",    "Unknown"),
                "categories":        ", ".join(cats.values()) if cats else "",
            }
        except Exception as exc:
            logger.error(f"VT error for {ip} attempt {attempt+1}: {exc}")
            time.sleep(5)
    return default


def run_full_pipeline(targets: list, vt_api_key: str, progress_cb=None) -> list:
    """
    Run the complete scan pipeline for a list of targets:
      1. Nmap scan each target
      2. VirusTotal enrichment per unique IP
      3. Attach risk scores and vulnerability metadata

    progress_cb(pct: float, message: str) - optional progress callback
    Returns a list of enriched row dicts ready for a DataFrame.
    """
    all_rows   = []
    n_targets  = len(targets)

    def _cb(pct, msg):
        if progress_cb:
            progress_cb(pct, msg)
        logger.info(f"[{pct:.0%}] {msg}")

    # Phase 1 - Nmap
    for i, tgt in enumerate(targets):
        _cb((i + 0.5) / (n_targets * 2), f"Nmap scanning {tgt} …")
        xml  = run_nmap(tgt)
        rows = parse_nmap_xml(xml)
        all_rows.extend(rows)
        _cb((i + 1) / (n_targets * 2), f"Nmap done for {tgt} - {len(rows)} port(s) found")

    if not all_rows:
        _cb(1.0, "Nmap found no open ports")
        return []

    # Phase 2 - VirusTotal
    import pandas as pd
    df         = pd.DataFrame(all_rows)
    unique_ips = df["ip"].unique().tolist()
    vt_cache   = {}
    n_total    = n_targets + len(unique_ips)

    for j, ip in enumerate(unique_ips):
        _cb((n_targets + j + 0.5) / n_total, f"VirusTotal checking {ip} …")
        vt_cache[ip] = check_virustotal(ip, vt_api_key)
        _cb((n_targets + j + 1) / n_total, f"VT done for {ip}")
        if j < len(unique_ips) - 1:
            time.sleep(16)   # free VT tier = 4 req/min

    # Phase 3 - Enrich rows
    enriched = []
    for row in all_rows:
        vt    = vt_cache.get(row["ip"], {})
        score = calc_risk(row["service"], vt.get("malicious_reports", 0))
        sev   = classify_severity(score)
        vuln  = get_vuln(row["service"])
        enriched.append({
            **row, **vt,
            "risk_score":    score,
            "severity":      sev,
            "vulnerability": vuln["name"],
            "cve_ref":       vuln["cve"],
            "cvss":          vuln["cvss"],
            "action":        vuln["action"],
        })

    _cb(1.0, f"Complete - {len(enriched)} finding(s) across {len(unique_ips)} host(s)")
    return enriched

Overwriting modules/scanner.py


In [6]:
%%writefile modules/database.py
# modules/database.py
# ------------------------------------------------------------------
# SQLite persistence layer
#   - Stores every scan session + all enriched rows
#   - Provides history retrieval, stats, and delete helpers
# ------------------------------------------------------------------
import sqlite3
import os
import logging
from datetime import datetime

logger  = logging.getLogger(__name__)
DB_PATH = os.path.join(os.path.dirname(__file__), "..", "netguard.db")


def _connect() -> sqlite3.Connection:
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn


def init_db():
    """Create database tables if they do not already exist."""
    with _connect() as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS scan_sessions (
                id          INTEGER PRIMARY KEY AUTOINCREMENT,
                started_at  TEXT    NOT NULL,
                targets     TEXT    NOT NULL,
                total_hosts INTEGER DEFAULT 0,
                total_ports INTEGER DEFAULT 0,
                critical_ct INTEGER DEFAULT 0,
                high_ct     INTEGER DEFAULT 0,
                max_risk    REAL    DEFAULT 0
            )
        """)
        conn.execute("""
            CREATE TABLE IF NOT EXISTS scan_records (
                id                INTEGER PRIMARY KEY AUTOINCREMENT,
                session_id        INTEGER NOT NULL,
                ip                TEXT,
                hostname          TEXT,
                port              TEXT,
                service           TEXT,
                product           TEXT,
                version           TEXT,
                malicious_reports INTEGER DEFAULT 0,
                suspicious_count  INTEGER DEFAULT 0,
                harmless_count    INTEGER DEFAULT 0,
                community_score   INTEGER DEFAULT 0,
                country           TEXT,
                network           TEXT,
                categories        TEXT,
                risk_score        REAL    DEFAULT 0,
                severity          TEXT,
                vulnerability     TEXT,
                cve_ref           TEXT,
                cvss              REAL    DEFAULT 0,
                action            TEXT,
                FOREIGN KEY (session_id) REFERENCES scan_sessions(id)
            )
        """)
        conn.commit()
    logger.info("Database initialised")


def save_scan(targets: list, rows: list) -> int:
    """Persist a completed scan session and all its rows. Returns the session ID."""
    import pandas as pd
    df = pd.DataFrame(rows) if rows else pd.DataFrame()

    started_at  = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    targets_str = ", ".join(targets)
    total_hosts = int(df["ip"].nunique())                       if not df.empty else 0
    total_ports = len(df)                                       if not df.empty else 0
    critical_ct = int((df["severity"] == "Critical").sum())     if not df.empty else 0
    high_ct     = int((df["severity"] == "High").sum())         if not df.empty else 0
    max_risk    = float(df["risk_score"].max())                  if not df.empty else 0.0

    with _connect() as conn:
        cur = conn.execute("""
            INSERT INTO scan_sessions
                (started_at, targets, total_hosts, total_ports, critical_ct, high_ct, max_risk)
            VALUES (?,?,?,?,?,?,?)
        """, (started_at, targets_str, total_hosts, total_ports, critical_ct, high_ct, max_risk))
        session_id = cur.lastrowid

        if rows:
            conn.executemany("""
                INSERT INTO scan_records
                    (session_id,ip,hostname,port,service,product,version,
                     malicious_reports,suspicious_count,harmless_count,
                     community_score,country,network,categories,
                     risk_score,severity,vulnerability,cve_ref,cvss,action)
                VALUES
                    (:session_id,:ip,:hostname,:port,:service,:product,:version,
                     :malicious_reports,:suspicious_count,:harmless_count,
                     :community_score,:country,:network,:categories,
                     :risk_score,:severity,:vulnerability,:cve_ref,:cvss,:action)
            """, [{**r, "session_id": session_id} for r in rows])
        conn.commit()

    logger.info(f"Saved session {session_id} with {len(rows)} records")
    return session_id


def get_sessions(limit: int = 50) -> list:
    """Return the most recent scan sessions (summary rows only)."""
    with _connect() as conn:
        rows = conn.execute(
            "SELECT * FROM scan_sessions ORDER BY id DESC LIMIT ?", (limit,)
        ).fetchall()
    return [dict(r) for r in rows]


def get_session_records(session_id: int) -> list:
    """Return all scan records for a given session ID."""
    with _connect() as conn:
        rows = conn.execute(
            "SELECT * FROM scan_records WHERE session_id=? ORDER BY risk_score DESC",
            (session_id,)
        ).fetchall()
    return [dict(r) for r in rows]


def get_all_records() -> list:
    """Return every record across all sessions joined with session metadata."""
    with _connect() as conn:
        rows = conn.execute("""
            SELECT r.*, s.started_at, s.targets
            FROM scan_records r
            JOIN scan_sessions s ON r.session_id = s.id
            ORDER BY r.id DESC
        """).fetchall()
    return [dict(r) for r in rows]


def delete_session(session_id: int):
    """Delete a session and all associated records."""
    with _connect() as conn:
        conn.execute("DELETE FROM scan_records  WHERE session_id=?", (session_id,))
        conn.execute("DELETE FROM scan_sessions WHERE id=?",         (session_id,))
        conn.commit()
    logger.info(f"Deleted session {session_id}")


def get_db_stats() -> dict:
    """Return quick summary counts from the database."""
    with _connect() as conn:
        sessions = conn.execute("SELECT COUNT(*) FROM scan_sessions").fetchone()[0]
        records  = conn.execute("SELECT COUNT(*) FROM scan_records").fetchone()[0]
        critical = conn.execute("SELECT COUNT(*) FROM scan_records WHERE severity='Critical'").fetchone()[0]
        high     = conn.execute("SELECT COUNT(*) FROM scan_records WHERE severity='High'").fetchone()[0]
    return {
        "total_sessions": sessions,
        "total_records":  records,
        "critical_total": critical,
        "high_total":     high,
    }

Overwriting modules/database.py


In [7]:
%%writefile modules/emailer.py
# modules/emailer.py - Email & PDF Report Module
import smtplib, logging, re, os
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from email.mime.base import MIMEBase
from email import encoders
from fpdf import FPDF

logger = logging.getLogger(__name__)
os.makedirs("reports", exist_ok=True)

SEV_COLORS = {"Critical": "#c0392b", "High": "#e67e22", "Medium": "#f39c12", "Low": "#27ae60"}

def is_valid_email(addr: str) -> bool:
    return bool(re.match(r"^[^@\s]+@[^@\s]+\.[^@\s]+$", addr.strip()))

def build_html_email(alert_df, scan_time: str, max_risk: int) -> str:
    crit_n = int((alert_df["severity"] == "Critical").sum())
    max_sev = "Critical" if crit_n > 0 else "High"
    hosts = ", ".join(sorted(alert_df["ip"].unique()))
    sc = SEV_COLORS.get(max_sev, "#c0392b")
    rows_html = ""
    for _, row in alert_df.iterrows():
        sev = row.get("severity", "Low")
        color = SEV_COLORS.get(sev, "#888")
        rows_html += f"<tr><td>{row['ip']}</td><td>{row.get('vulnerability','')}</td><td style='color:{color};font-weight:bold;'>{sev}</td></tr>"
    return f"<html><body style='font-family:Arial;'><h2>NetGuard Alert</h2><p>Findings: {len(alert_df)}, Risk: {max_risk}/10, Time: {scan_time}</p><table border='1'><tr><th>IP</th><th>Vulnerability</th><th>Severity</th></tr>{rows_html}</table></body></html>"

def build_pdf_report(alert_df, scan_time: str, max_risk: int) -> bytes:
    pdf = FPDF()
    pdf.add_page()
    pdf.set_font("Helvetica", "B", 14)
    pdf.cell(0, 10, "NetGuard Report", ln=True, align="C")
    pdf.set_font("Helvetica", "", 10)
    pdf.cell(0, 10, f"Time: {scan_time} | Count: {len(alert_df)} | Max: {max_risk}", ln=True)
    pdf.ln(5)
    pdf.set_font("Helvetica", "B", 9)
    pdf.cell(40, 8, "IP"), pdf.cell(35, 8, "Vulnerability"), pdf.cell(20, 8, "Severity"), pdf.ln()
    pdf.set_font("Helvetica", "", 8)
    for _, row in alert_df.iterrows():
        ip = str(row.get("ip", ""))[:20]
        vuln = str(row.get("vulnerability", ""))[:20]
        sev = str(row.get("severity", ""))[:10]
        pdf.cell(40, 7, ip), pdf.cell(35, 7, vuln), pdf.cell(20, 7, sev), pdf.ln()
    return pdf.output(dest='S').encode('latin-1')

def send_alert_email(sender: str, password: str, recipient: str, alert_df, scan_time: str, max_risk: int):
    if not is_valid_email(sender) or not is_valid_email(recipient):
        return "Invalid email"
    subject = f"NetGuard - {len(alert_df)} findings"
    msg = MIMEMultipart("mixed")
    msg["From"], msg["To"], msg["Subject"] = sender, recipient, subject
    msg.attach(MIMEText(build_html_email(alert_df, scan_time, max_risk), "html"))
    pdf_bytes = build_pdf_report(alert_df, scan_time, max_risk)
    part = MIMEBase("application", "octet-stream")
    part.set_payload(pdf_bytes)
    encoders.encode_base64(part)
    part.add_header("Content-Disposition", "attachment; filename=Report.pdf")
    msg.attach(part)
    try:
        srv = smtplib.SMTP("smtp.gmail.com", 587)
        srv.starttls(), srv.login(sender, password), srv.send_message(msg), srv.quit()
        return True
    except:
        return "Failed"

Overwriting modules/emailer.py


In [8]:
%%writefile dashboard/app.py
# dashboard/app.py
# ------------------------------------------------------------------
# NetGuard - Main Dashboard (Overview page)
# Run: streamlit run dashboard/app.py
# ------------------------------------------------------------------
import sys, os
ROOT = os.path.abspath(os.path.join(os.path.dirname(__file__), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import streamlit as st
import pandas    as pd
import plotly.express       as px
import plotly.graph_objects as go
from datetime import datetime
from dotenv   import load_dotenv

from modules.scanner  import run_full_pipeline, is_valid_target
from modules.database import init_db, save_scan, get_db_stats
from modules.emailer  import send_alert_email, build_pdf_report

load_dotenv(os.path.join(ROOT, ".env"))

# ── Page config ────────────────────────────────────────────────
st.set_page_config(
    page_title="NetGuard",
    page_icon="🛡️",
    layout="wide",
    initial_sidebar_state="expanded",
)
init_db()

# ── Credentials ────────────────────────────────────────────────
VT_API_KEY      = os.environ.get("VT_API_KEY",      "")
GMAIL_SENDER    = os.environ.get("GMAIL_SENDER",    "")
GMAIL_PASSWORD  = os.environ.get("GMAIL_PASSWORD",  "")
GMAIL_RECIPIENT = os.environ.get("GMAIL_RECIPIENT", "")
TARGETS_ENV     = os.environ.get("SCAN_TARGETS",    "")

DEFAULT_TARGETS = "testphp.vulnweb.com,testasp.vulnweb.com"
DEFAULT_LIST    = [t.strip() for t in (TARGETS_ENV or DEFAULT_TARGETS).split(",") if t.strip()]

vt_ok    = bool(VT_API_KEY and not VT_API_KEY.startswith("your_"))
email_ok = bool(GMAIL_SENDER and GMAIL_PASSWORD and GMAIL_RECIPIENT
                and not GMAIL_SENDER.startswith("your_"))

# ── Session state defaults ─────────────────────────────────────
for k, v in [("scan_df", None), ("scan_time", None),
             ("session_id", None), ("targets_used", None)]:
    if k not in st.session_state:
        st.session_state[k] = v

# ── Global CSS ─────────────────────────────────────────────────
st.markdown("""
<style>
[data-testid="stAppViewContainer"] { background:#0d0d1a; color:#e2e8f0; }
[data-testid="stSidebar"]          { background:#12122a; border-right:2px solid #4a1d96; }
[data-testid="stSidebar"] *        { color:#c4b5fd !important; }
h1 { color:#a78bfa !important; }
h2 { color:#7c3aed !important; }
h3 { color:#6d28d9 !important; }
[data-testid="stMetric"] {
    background:linear-gradient(135deg,#1a1a3e,#2d1b69);
    border:1px solid #4a1d96; border-radius:14px; padding:14px;
}
[data-testid="stMetricValue"] { color:#fbbf24 !important; font-size:2rem !important; }
[data-testid="stMetricLabel"] { color:#a78bfa !important; font-size:0.8rem !important; }
.stButton>button {
    background:linear-gradient(135deg,#4a1d96,#7c3aed) !important;
    border:none !important; color:#fff !important;
    border-radius:10px; font-weight:700;
}
[data-testid="stDownloadButton"]>button {
    background:linear-gradient(135deg,#065f46,#059669) !important;
    border:none !important; color:#fff !important; border-radius:10px;
}
input, textarea {
    background:#1a1a3e !important; color:#e2e8f0 !important;
    border:1px solid #4a1d9688 !important; border-radius:8px;
}
hr { border-color:#4a1d9633 !important; }
::-webkit-scrollbar { width:6px; height:6px; }
::-webkit-scrollbar-thumb { background:#4a1d96; border-radius:3px; }
</style>
""", unsafe_allow_html=True)

# ── Sidebar ────────────────────────────────────────────────────
with st.sidebar:
    st.markdown("## 🛡️ NetGuard")
    st.markdown("*Cyber Risk Assessment Platform*")
    st.divider()

    # Credential status
    st.markdown("**⚙️ Status**")
    st.caption(f"{'✅' if vt_ok    else '❌'} VirusTotal API")
    st.caption(f"{'✅' if email_ok else '❌'} Email alerts")
    st.divider()

    # Target input
    st.markdown("**🎯 Scan Targets**")
    st.caption("One target per line, or comma-separated:")
    target_input = st.text_area(
        "Targets", label_visibility="collapsed",
        value="",
        placeholder="testphp.vulnweb.com\ntestasp.vulnweb.com\n192.168.1.1",
        height=100,
    )

    st.markdown("**Quick targets:**")
    QUICK = [
        "testphp.vulnweb.com",
        "testasp.vulnweb.com",
        "testaspnet.vulnweb.com",
        "zero.webappsecurity.com",
        "pentest-ground.com",
        "demo.testfire.net",
        "demo.owasp-juice.shop",
        "scanme.nmap.org",
    ]
    for t in QUICK:
        st.caption(f"  • `{t}`")
    st.caption("⚠️ Only scan authorised targets!")

    # Parse targets
    raw_targets = [
        t.strip()
        for part in (target_input or "").replace(",", "\n").splitlines()
        for t in [part.strip()] if t.strip()
    ]
    if raw_targets:
        invalid = [t for t in raw_targets if not is_valid_target(t)]
        if invalid:
            st.warning(f"Invalid target(s) skipped: {', '.join(invalid)}")
        active_targets = [t for t in raw_targets if is_valid_target(t)] or DEFAULT_LIST
    else:
        active_targets = DEFAULT_LIST

    st.divider()
    st.markdown("**🚀 Controls**")
    scan_btn    = st.button("🚀 Run Scan",        use_container_width=True, type="primary")
    refresh_btn = st.button("🔄 Refresh / Reset", use_container_width=True)

    if refresh_btn:
        for k in ("scan_df", "scan_time", "session_id", "targets_used"):
            st.session_state[k] = None
        st.rerun()

    st.divider()
    db = get_db_stats()
    st.markdown("**📊 Database**")
    st.caption(f"Sessions : {db['total_sessions']}")
    st.caption(f"Records  : {db['total_records']}")
    st.caption(f"Critical : {db['critical_total']}")

# ── Run scan ───────────────────────────────────────────────────
if scan_btn:
    if not vt_ok:
        st.error("❌ VT_API_KEY not set. Open .env, add your VirusTotal API key, restart.")
    else:
        bar    = st.progress(0)
        status = st.empty()

        def _cb(pct, msg):
            bar.progress(min(float(pct), 1.0))
            status.info(f"⏳ {msg}")

        rows = run_full_pipeline(active_targets, VT_API_KEY, _cb)
        bar.empty()

        if not rows:
            status.warning(
                "Nmap returned no open ports. "
                "Try: (1) run as Administrator  "
                "(2) check nmap --version in terminal  "
                "(3) confirm target is reachable."
            )
        else:
            df = pd.DataFrame(rows)
            st.session_state.scan_df      = df
            st.session_state.scan_time    = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            st.session_state.targets_used = active_targets
            sid = save_scan(active_targets, rows)
            st.session_state.session_id   = sid
            status.success(
                f"✅ Scan complete - {len(df)} finding(s) across {df['ip'].nunique()} host(s) "
                f"| Session #{sid} saved"
            )

            # Auto-send email if High/Critical found and email is configured
            alert_df = df[df["severity"].isin(["High", "Critical"])]
            if not alert_df.empty and email_ok:
                with st.spinner("📧 Sending alert email …"):
                    res = send_alert_email(
                        GMAIL_SENDER, GMAIL_PASSWORD, GMAIL_RECIPIENT,
                        alert_df, st.session_state.scan_time, int(df["risk_score"].max())
                    )
                if res is True:
                    st.success(f"📧 Alert email + PDF sent to {GMAIL_RECIPIENT}")
                else:
                    st.warning(f"⚠️ Email failed: {res}")

# ── Hero header ────────────────────────────────────────────────
st.markdown("""
<div style='background:linear-gradient(135deg,#1a1a3e 0%,#2d1b69 60%,#1a1a3e 100%);
            border:1px solid #4a1d96;border-radius:16px;padding:28px 36px;margin-bottom:24px;'>
  <div style='display:flex;align-items:center;gap:16px;'>
    <span style='font-size:44px;'>🛡️</span>
    <div>
      <h1 style='margin:0;color:#a78bfa;font-size:2.2rem;font-weight:800;'>NetGuard</h1>
      <p style='margin:4px 0 0;color:#9ca3af;font-size:14px;'>
        Cyber Risk Assessment &amp; Threat Intelligence Platform
      </p>
    </div>
    <div style='margin-left:auto;'>
      <span style='background:#4a1d96;color:#c4b5fd;padding:6px 16px;
                   border-radius:20px;font-size:12px;font-weight:600;'>🏠 Overview</span>
    </div>
  </div>
</div>
""", unsafe_allow_html=True)

# Scan time strip
if st.session_state.scan_time:
    c1, c2 = st.columns([8, 1])
    c1.info(
        f"🕐 Last scan: {st.session_state.scan_time}  |  "
        f"Session #{st.session_state.session_id}  |  "
        f"Targets: {', '.join(st.session_state.targets_used or [])}"
    )
    if c2.button("🔄", help="Reset"):
        for k in ("scan_df", "scan_time", "session_id", "targets_used"):
            st.session_state[k] = None
        st.rerun()
else:
    st.info("👋 No scan run yet. Enter a target in the sidebar and click **Run Scan**.")

st.divider()

df = st.session_state.scan_df
BG, GRID, FONT = "#12122a", "#1a1a3e", "#e2e8f0"
SEV_C = {"Critical":"#e74c3c","High":"#e67e22","Medium":"#f1c40f",
          "Low":"#27ae60","Informational":"#2980b9"}

if df is None or df.empty:
    # Welcome cards
    st.markdown("""
    <div style='text-align:center;padding:60px 20px;'>
      <div style='font-size:64px;'>🛡️</div>
      <h2 style='color:#a78bfa;'>Welcome to NetGuard</h2>
      <p style='color:#9ca3af;max-width:500px;margin:0 auto 32px;font-size:15px;line-height:1.7;'>
        Enter a target in the sidebar and click <strong style='color:#a78bfa;'>Run Scan</strong>
        to begin. Results will appear here with full risk analysis.
      </p>
      <div style='display:flex;gap:20px;justify-content:center;flex-wrap:wrap;'>
        <div style='background:#1a1a3e;border:1px solid #4a1d96;
                    border-radius:12px;padding:20px 28px;'>
          <div style='font-size:28px;'>🔍</div>
          <div style='color:#a78bfa;font-weight:600;margin-top:8px;'>Port Scanning</div>
          <div style='color:#6b7280;font-size:12px;'>65+ ports via Nmap</div>
        </div>
        <div style='background:#1a1a3e;border:1px solid #4a1d96;
                    border-radius:12px;padding:20px 28px;'>
          <div style='font-size:28px;'>🌐</div>
          <div style='color:#a78bfa;font-weight:600;margin-top:8px;'>Threat Intel</div>
          <div style='color:#6b7280;font-size:12px;'>VirusTotal enrichment</div>
        </div>
        <div style='background:#1a1a3e;border:1px solid #4a1d96;
                    border-radius:12px;padding:20px 28px;'>
          <div style='font-size:28px;'>📊</div>
          <div style='color:#a78bfa;font-weight:600;margin-top:8px;'>Risk Scoring</div>
          <div style='color:#6b7280;font-size:12px;'>CVSS-based analysis</div>
        </div>
        <div style='background:#1a1a3e;border:1px solid #4a1d96;
                    border-radius:12px;padding:20px 28px;'>
          <div style='font-size:28px;'>📧</div>
          <div style='color:#a78bfa;font-weight:600;margin-top:8px;'>Auto Alerts</div>
          <div style='color:#6b7280;font-size:12px;'>Email + PDF reports</div>
        </div>
      </div>
    </div>
    """, unsafe_allow_html=True)

else:
    # ── KPI Metrics ────────────────────────────────────────────
    st.markdown("### 📊 Key Metrics")
    hc = len(df[df["severity"].isin(["High","Critical"])])
    m1,m2,m3,m4,m5,m6 = st.columns(6)
    m1.metric("🖥 Hosts",         df["ip"].nunique())
    m2.metric("🔓 Open Ports",    len(df))
    m3.metric("⚙️ Services",      df["service"].nunique())
    m4.metric("💀 Max Risk",      int(df["risk_score"].max()))
    m5.metric("🚨 High+Critical", hc)
    m6.metric("🦠 Max VT Hits",   int(df["malicious_reports"].max()))
    st.divider()

    # ── Quick charts ───────────────────────────────────────────
    st.markdown("### 🗺 Quick Overview")
    oc1, oc2, oc3 = st.columns(3)

    with oc1:
        sev_order = ["Critical","High","Medium","Low","Informational"]
        counts    = df["severity"].value_counts().reindex(sev_order).dropna()
        fig = go.Figure(go.Pie(
            labels=counts.index, values=counts.values, hole=0.55,
            marker=dict(colors=[SEV_C[s] for s in counts.index],
                        line=dict(color=BG, width=2)),
            textinfo="label+percent", textfont=dict(color=FONT, size=10),
        ))
        fig.update_layout(
            title=dict(text="Severity Split", font=dict(color="#a78bfa", size=12)),
            paper_bgcolor=BG, height=280, margin=dict(l=0,r=0,t=36,b=0),
            showlegend=False, font=dict(color=FONT),
        )
        st.plotly_chart(fig, use_container_width=True)

    with oc2:
        pc = df.groupby("ip")["port"].count().reset_index()
        pc.columns = ["IP", "Ports"]
        fig = px.bar(pc, x="IP", y="Ports", color="Ports",
                     color_continuous_scale=[[0,"#1a1a3e"],[0.5,"#7c3aed"],[1,"#fbbf24"]],
                     text="Ports")
        fig.update_traces(textposition="outside", marker_line_width=1)
        fig.update_layout(
            title=dict(text="Ports per Host", font=dict(color="#a78bfa", size=12)),
            paper_bgcolor=BG, plot_bgcolor=GRID, height=280,
            showlegend=False, font=dict(color=FONT),
            margin=dict(l=0,r=0,t=36,b=40),
            xaxis=dict(gridcolor=GRID, tickfont=dict(size=9)),
            yaxis=dict(gridcolor=GRID),
        )
        st.plotly_chart(fig, use_container_width=True)

    with oc3:
        rs = df.groupby("ip")["risk_score"].max().reset_index()
        rs.columns = ["IP","Max Risk"]
        bar_c = ["#e74c3c" if v>=9 else "#e67e22" if v>=7 else "#f1c40f" if v>=4 else "#27ae60"
                 for v in rs["Max Risk"]]
        fig = go.Figure(go.Bar(
            x=rs["IP"], y=rs["Max Risk"],
            marker_color=bar_c,
            text=rs["Max Risk"], textposition="outside",
        ))
        fig.update_layout(
            title=dict(text="Max Risk per Host", font=dict(color="#a78bfa", size=12)),
            paper_bgcolor=BG, plot_bgcolor=GRID, height=280,
            showlegend=False, font=dict(color=FONT),
            margin=dict(l=0,r=0,t=36,b=40),
            xaxis=dict(gridcolor=GRID, tickfont=dict(size=9)),
            yaxis=dict(gridcolor=GRID, range=[0,11]),
        )
        st.plotly_chart(fig, use_container_width=True)

    st.divider()

    # ── Host risk summary ──────────────────────────────────────
    st.markdown("### 📌 Host Risk Summary")
    summary = df.groupby("ip").agg(
        hostname       = ("hostname",          "first"),
        total_ports    = ("port",               "count"),
        services       = ("service",            lambda x: ", ".join(sorted(x.unique()))),
        malicious_score= ("malicious_reports",  "max"),
        max_risk       = ("risk_score",         "max"),
        severity       = ("severity",           lambda x: x.value_counts().index[0]),
    ).reset_index()
    summary.columns = ["IP","Hostname","Ports","Services","VT Malicious","Max Risk","Top Severity"]
    summary = summary.sort_values("Max Risk", ascending=False)
    st.dataframe(summary, use_container_width=True, hide_index=True)
    st.divider()

    # ── Quick actions ──────────────────────────────────────────
    st.markdown("### 📧 Quick Actions")
    alert_df = df[df["severity"].isin(["Critical","High"])]
    qa1, qa2, qa3 = st.columns(3)

    with qa1:
        if not email_ok:
            st.warning("Email not configured in .env")
        else:
            if st.button(f"📧 Send Alert Email ({len(alert_df)} findings)",
                         use_container_width=True, disabled=alert_df.empty):
                with st.spinner("Sending …"):
                    res = send_alert_email(
                        GMAIL_SENDER, GMAIL_PASSWORD, GMAIL_RECIPIENT,
                        alert_df if not alert_df.empty else df.head(5),
                        st.session_state.scan_time or datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                        int(df["risk_score"].max()),
                    )
                if res is True:
                    st.success(f"Sent to {GMAIL_RECIPIENT}")
                else:
                    st.error(f"Failed: {res}")

    with qa2:
        _ae  = alert_df if not alert_df.empty else df.head(5)
        _ts  = st.session_state.scan_time or datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        _pdf = build_pdf_report(_ae, _ts, int(df["risk_score"].max()))
        st.download_button("⬇️ Download PDF Report", data=_pdf,
                           file_name="NetGuard_Report.pdf", mime="application/pdf",
                           use_container_width=True)

    with qa3:
        st.download_button("⬇️ Download CSV (Full)",
                           data=df.to_csv(index=False).encode("utf-8"),
                           file_name="netguard_scan.csv", mime="text/csv",
                           use_container_width=True)

Overwriting dashboard/app.py


In [9]:
%%writefile dashboard/pages/2_Scan_Data.py
# dashboard/pages/2_Scan_Data.py
# Page 2 - Full colour-coded scan results with filters
import sys, os
ROOT = os.path.abspath(os.path.join(os.path.dirname(__file__), "..", ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import streamlit as st
import pandas    as pd
from dotenv import load_dotenv
load_dotenv(os.path.join(ROOT, ".env"))

st.set_page_config(page_title="Scan Data | NetGuard", page_icon="📋", layout="wide")

SEV_C = {"Critical":"#e74c3c","High":"#e67e22",
          "Medium":"#f1c40f","Low":"#27ae60","Informational":"#2980b9"}

st.markdown("""
<style>
[data-testid="stAppViewContainer"] { background:#0d0d1a; color:#e2e8f0; }
[data-testid="stSidebar"] { background:#12122a; border-right:2px solid #4a1d96; }
[data-testid="stSidebar"] * { color:#c4b5fd !important; }
h1 { color:#a78bfa !important; }
[data-testid="stMetric"] { background:linear-gradient(135deg,#1a1a3e,#2d1b69);
    border:1px solid #4a1d96; border-radius:14px; padding:14px; }
[data-testid="stMetricValue"] { color:#fbbf24 !important; }
[data-testid="stMetricLabel"] { color:#a78bfa !important; }
</style>
""", unsafe_allow_html=True)

st.markdown("""
<div style='background:linear-gradient(135deg,#1a1a3e,#2d1b69);
            border:1px solid #4a1d96;border-radius:14px;padding:22px 30px;margin-bottom:20px;'>
  <h1 style='margin:0;color:#a78bfa;'>📋 Scan Data</h1>
  <p style='margin:4px 0 0;color:#9ca3af;font-size:13px;'>
    Full colour-coded results with filters
  </p>
</div>
""", unsafe_allow_html=True)

df = st.session_state.get("scan_df")
if df is None or df.empty:
    st.info("No scan data yet. Run a scan from the **🏠 Overview** page.")
    st.stop()

# Sidebar filters
with st.sidebar:
    st.markdown("**🔍 Filters**")
    sel_ip  = st.selectbox("IP",      ["All"] + sorted(df["ip"].unique().tolist()))
    sel_svc = st.selectbox("Service", ["All"] + sorted(df["service"].unique().tolist()))
    sel_sev = st.multiselect(
        "Severity",
        ["Critical","High","Medium","Low","Informational"],
        default=["Critical","High","Medium","Low","Informational"]
    )
    mn = int(df["risk_score"].min()); mx = int(df["risk_score"].max())
    if mn == mx: mx = mn + 1
    min_risk = st.slider("Min Risk Score", mn, mx, mn)
    search   = st.text_input("🔎 Search", "")

filt = df.copy()
if sel_ip  != "All": filt = filt[filt["ip"]      == sel_ip]
if sel_svc != "All": filt = filt[filt["service"] == sel_svc]
if sel_sev:          filt = filt[filt["severity"].isin(sel_sev)]
filt = filt[filt["risk_score"] >= min_risk]
if search.strip():
    mask = filt.apply(lambda r: search.lower() in str(r).lower(), axis=1)
    filt = filt[mask]

c1,c2,c3,c4 = st.columns(4)
c1.metric("Showing",      f"{len(filt)} / {len(df)}")
c2.metric("Hosts",        filt["ip"].nunique())
c3.metric("Critical+High",len(filt[filt["severity"].isin(["Critical","High"])]))
c4.metric("Avg Risk",     round(filt["risk_score"].mean(), 1) if not filt.empty else 0)
st.divider()

cols = ["ip","hostname","port","service","product","version",
        "risk_score","severity","vulnerability","cve_ref","cvss",
        "malicious_reports","country","action"]
cols = [c for c in cols if c in filt.columns]

def _color_sev(val):
    c = SEV_C.get(str(val), "")
    return f"background-color:{c}33;color:{c};font-weight:bold;" if c else ""

def _color_risk(val):
    try:
        v = float(val)
        c = "#e74c3c" if v>=9 else "#e67e22" if v>=7 else "#f1c40f" if v>=4 else "#27ae60"
        return f"color:{c};font-weight:bold;"
    except:
        return ""

styled = filt[cols].style
if "severity"   in cols: styled = styled.map(_color_sev,  subset=["severity"])
if "risk_score" in cols: styled = styled.map(_color_risk, subset=["risk_score"])

st.dataframe(styled, use_container_width=True, hide_index=True, height=480)

Overwriting dashboard/pages/2_Scan_Data.py


In [10]:
%%writefile dashboard/pages/3_Charts.py
# dashboard/pages/3_Charts.py
# Page 3 - Interactive Plotly charts
import sys, os
ROOT = os.path.abspath(os.path.join(os.path.dirname(__file__), "..", ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import streamlit            as st
import plotly.express       as px
import plotly.graph_objects as go
import pandas               as pd
from dotenv import load_dotenv
load_dotenv(os.path.join(ROOT, ".env"))

st.set_page_config(page_title="Charts | NetGuard", page_icon="📈", layout="wide")

BG   = "#0d0d1a"
PAP  = "#12122a"
GRID = "#1a1a3e"
FONT = "#e2e8f0"
SEV_C = {"Critical":"#e74c3c","High":"#e67e22",
          "Medium":"#f1c40f","Low":"#27ae60","Informational":"#2980b9"}

st.markdown("""
<style>
[data-testid="stAppViewContainer"] { background:#0d0d1a; color:#e2e8f0; }
[data-testid="stSidebar"] { background:#12122a; border-right:2px solid #4a1d96; }
[data-testid="stSidebar"] * { color:#c4b5fd !important; }
h1 { color:#a78bfa !important; } h3 { color:#a78bfa !important; }
</style>
""", unsafe_allow_html=True)

def _lay(fig, title="", h=380):
    fig.update_layout(
        title=dict(text=title, font=dict(color="#a78bfa", size=13)),
        paper_bgcolor=PAP, plot_bgcolor=GRID, font=dict(color=FONT),
        xaxis=dict(gridcolor=GRID, zerolinecolor=GRID, tickfont=dict(size=9)),
        yaxis=dict(gridcolor=GRID, zerolinecolor=GRID),
        legend=dict(bgcolor=PAP, bordercolor="#4a1d96", font=dict(size=10)),
        margin=dict(l=36,r=16,t=44,b=36), height=h,
    )
    return fig

st.markdown("""
<div style='background:linear-gradient(135deg,#1a1a3e,#2d1b69);
            border:1px solid #4a1d96;border-radius:14px;padding:22px 30px;margin-bottom:20px;'>
  <h1 style='margin:0;color:#a78bfa;'>📈 Charts &amp; Analytics</h1>
  <p style='margin:4px 0 0;color:#9ca3af;font-size:13px;'>
    Interactive Plotly visualisations - hover, zoom, click legend to toggle
  </p>
</div>
""", unsafe_allow_html=True)

df = st.session_state.get("scan_df")
if df is None or df.empty:
    st.info("No scan data yet. Run a scan from the **🏠 Overview** page.")
    st.stop()

sev_order = ["Critical","High","Medium","Low","Informational"]

# Row 1 - Severity donut + Ports per host
r1c1, r1c2 = st.columns(2)
with r1c1:
    counts = df["severity"].value_counts().reindex(sev_order).dropna()
    fig = go.Figure(go.Pie(
        labels=counts.index, values=counts.values, hole=0.55,
        marker=dict(colors=[SEV_C[s] for s in counts.index],
                    line=dict(color=BG, width=2)),
        textinfo="label+percent+value",
        textfont=dict(color=FONT, size=11),
    ))
    st.plotly_chart(_lay(fig, "Severity Distribution", 360), use_container_width=True)

with r1c2:
    pc = df.groupby("ip")["port"].count().reset_index()
    pc.columns = ["IP","Ports"]
    pc = pc.sort_values("Ports", ascending=False)
    fig = px.bar(pc, x="IP", y="Ports",
                 color="Ports",
                 color_continuous_scale=[[0,"#1a1a3e"],[0.5,"#7c3aed"],[1,"#fbbf24"]],
                 text="Ports")
    fig.update_traces(textposition="outside")
    st.plotly_chart(_lay(fig, "Open Ports per Host", 360), use_container_width=True)

# Row 2 - Risk heatmap + Scatter risk vs ports
r2c1, r2c2 = st.columns(2)
with r2c1:
    pivot = df.pivot_table(
        index="ip", columns="service",
        values="risk_score", aggfunc="max",
    ).fillna(0)
    fig = px.imshow(
        pivot, color_continuous_scale="RdYlGn_r",
        labels=dict(x="Service", y="Host", color="Risk"),
        aspect="auto",
    )
    st.plotly_chart(_lay(fig, "Host × Service Risk Heatmap", 360), use_container_width=True)

with r2c2:
    host_agg = df.groupby("ip").agg(
        ports   = ("port",       "count"),
        risk    = ("risk_score", "max"),
        sev     = ("severity",   lambda x: x.value_counts().index[0]),
        malicious=("malicious_reports","max"),
    ).reset_index()
    fig = px.scatter(
        host_agg, x="ports", y="risk",
        size="malicious",
        color="sev",
        color_discrete_map=SEV_C,
        text="ip",
        labels={"ports":"Open Ports","risk":"Max Risk Score","sev":"Severity"},
        size_max=40,
    )
    fig.update_traces(textposition="top center")
    st.plotly_chart(_lay(fig, "Risk Score vs Open Ports", 360), use_container_width=True)

# Row 3 - CVE/CVSS bar + VT stacked bar
r3c1, r3c2 = st.columns(2)
with r3c1:
    cvss = df[["service","cvss"]].drop_duplicates().sort_values("cvss", ascending=False).head(15)
    fig  = px.bar(cvss, x="cvss", y="service", orientation="h",
                  color="cvss",
                  color_continuous_scale=[[0,"#27ae60"],[0.5,"#f1c40f"],[1,"#e74c3c"]],
                  text="cvss")
    fig.update_traces(texttemplate="%{text:.1f}", textposition="outside")
    st.plotly_chart(_lay(fig, "CVSS Score by Service (Top 15)", 360), use_container_width=True)

with r3c2:
    vt = df.groupby("ip").agg(
        malicious  = ("malicious_reports", "max"),
        suspicious = ("suspicious_count",  "max"),
        harmless   = ("harmless_count",    "max"),
    ).reset_index()
    fig = go.Figure()
    for col, c, lbl in [
        ("malicious",  "#e74c3c", "Malicious"),
        ("suspicious", "#e67e22", "Suspicious"),
        ("harmless",   "#27ae60", "Harmless"),
    ]:
        fig.add_trace(go.Bar(x=vt["ip"], y=vt[col],
                             name=lbl, marker_color=c))
    fig.update_layout(barmode="stack")
    st.plotly_chart(_lay(fig, "VirusTotal Reports per Host", 360), use_container_width=True)

Overwriting dashboard/pages/3_Charts.py


In [11]:
%%writefile dashboard/pages/4_Threat_Intel.py
# dashboard/pages/4_Threat_Intel.py
# Page 4 - Threat Intelligence deep-dive
import sys, os
ROOT = os.path.abspath(os.path.join(os.path.dirname(__file__), "..", ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import streamlit as st
import pandas    as pd
import plotly.graph_objects as go
import plotly.express       as px
from dotenv import load_dotenv
load_dotenv(os.path.join(ROOT, ".env"))

st.set_page_config(page_title="Threat Intel | NetGuard", page_icon="🚨", layout="wide")
BG = "#0d0d1a"; PAP = "#12122a"; GRID = "#1a1a3e"; FONT = "#e2e8f0"
SEV_C = {"Critical":"#e74c3c","High":"#e67e22",
          "Medium":"#f1c40f","Low":"#27ae60","Informational":"#2980b9"}

st.markdown("""
<style>
[data-testid="stAppViewContainer"] { background:#0d0d1a; color:#e2e8f0; }
[data-testid="stSidebar"] { background:#12122a; border-right:2px solid #4a1d96; }
[data-testid="stSidebar"] * { color:#c4b5fd !important; }
h1, h2 { color:#a78bfa !important; }
</style>
""", unsafe_allow_html=True)

st.markdown("""
<div style='background:linear-gradient(135deg,#1a1a3e,#2d1b69);
            border:1px solid #4a1d96;border-radius:14px;padding:22px 30px;margin-bottom:20px;'>
  <h1 style='margin:0;color:#a78bfa;'>🚨 Threat Intelligence</h1>
  <p style='margin:4px 0 0;color:#9ca3af;font-size:13px;'>
    VirusTotal data, CVE references &amp; remediation actions
  </p>
</div>
""", unsafe_allow_html=True)

df = st.session_state.get("scan_df")
if df is None or df.empty:
    st.info("No scan data yet. Run a scan from the **🏠 Overview** page.")
    st.stop()

# Top critical findings
st.markdown("### 🔴 Critical & High Findings")
alert_df = df[df["severity"].isin(["Critical","High"])].sort_values("risk_score", ascending=False)
if alert_df.empty:
    st.success("✅ No Critical or High findings detected.")
else:
    cols = ["ip","port","service","vulnerability","severity",
            "risk_score","cve_ref","cvss","malicious_reports","country","action"]
    cols = [c for c in cols if c in alert_df.columns]
    st.dataframe(alert_df[cols], use_container_width=True, hide_index=True)
st.divider()

# CVE reference cards
st.markdown("### 📄 CVE & Vulnerability Reference")
cve_df = df[["service","vulnerability","cve_ref","cvss","action"]].drop_duplicates()
cve_df = cve_df.sort_values("cvss", ascending=False)
for _, row in cve_df.iterrows():
    cvss  = float(row.get("cvss", 0))
    color = ("#e74c3c" if cvss >= 9 else "#e67e22" if cvss >= 7
             else "#f1c40f" if cvss >= 4 else "#27ae60")
    with st.expander(f"[{row['cve_ref']}]  {row['vulnerability']}  -  CVSS {cvss}"):
        c1, c2 = st.columns([1, 3])
        c1.metric("CVSS Score", f"{cvss}/10")
        c2.markdown(f"**Service:** `{row['service']}`  \\ **CVE:** `{row['cve_ref']}`")
        st.info(f"🔧 **Action:** {row['action']}")
st.divider()

# Country map
if "country" in df.columns:
    st.markdown("### 🌍 Geographic Distribution")
    geo = df[df["country"] != "Unknown"].groupby("country").agg(
        count=("ip", "count"), max_risk=("risk_score","max")
    ).reset_index()
    if not geo.empty:
        fig = px.choropleth(
            geo, locations="country", locationmode="ISO-3",
            color="max_risk",
            hover_name="country",
            hover_data=["count","max_risk"],
            color_continuous_scale="RdYlGn_r",
            title="Max Risk Score by Country",
        )
        fig.update_layout(
            paper_bgcolor=PAP, plot_bgcolor=PAP,
            font=dict(color=FONT), height=420,
            geo=dict(bgcolor=BG, lakecolor=BG, landcolor="#1a1a3e",
                     showframe=False, showcoastlines=True,
                     coastlinecolor="#4a1d96"),
        )
        st.plotly_chart(fig, use_container_width=True)

Overwriting dashboard/pages/4_Threat_Intel.py


In [12]:
%%writefile dashboard/pages/5_History.py
# dashboard/pages/5_History.py
# Page 5 - Scan history from SQLite database
import sys, os
ROOT = os.path.abspath(os.path.join(os.path.dirname(__file__), "..", ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import streamlit as st
import pandas    as pd
import plotly.express as px
from dotenv import load_dotenv
load_dotenv(os.path.join(ROOT, ".env"))

from modules.database import get_sessions, get_session_records, delete_session, get_db_stats

st.set_page_config(page_title="History | NetGuard", page_icon="🗂️", layout="wide")

st.markdown("""
<style>
[data-testid="stAppViewContainer"] { background:#0d0d1a; color:#e2e8f0; }
[data-testid="stSidebar"] { background:#12122a; border-right:2px solid #4a1d96; }
[data-testid="stSidebar"] * { color:#c4b5fd !important; }
h1 { color:#a78bfa !important; }
[data-testid="stMetric"] { background:linear-gradient(135deg,#1a1a3e,#2d1b69);
    border:1px solid #4a1d96; border-radius:14px; padding:14px; }
[data-testid="stMetricValue"] { color:#fbbf24 !important; }
[data-testid="stMetricLabel"] { color:#a78bfa !important; }
</style>
""", unsafe_allow_html=True)

st.markdown("""
<div style='background:linear-gradient(135deg,#1a1a3e,#2d1b69);
            border:1px solid #4a1d96;border-radius:14px;padding:22px 30px;margin-bottom:20px;'>
  <h1 style='margin:0;color:#a78bfa;'>🗂️ Scan History</h1>
  <p style='margin:4px 0 0;color:#9ca3af;font-size:13px;'>
    All previous scan sessions stored in the local database
  </p>
</div>
""", unsafe_allow_html=True)

stats = get_db_stats()
m1,m2,m3,m4 = st.columns(4)
m1.metric("Total Sessions",  stats["total_sessions"])
m2.metric("Total Records",   stats["total_records"])
m3.metric("Critical Total",  stats["critical_total"])
m4.metric("High Total",      stats["high_total"])
st.divider()

sessions = get_sessions(limit=50)
if not sessions:
    st.info("No scan sessions found. Run a scan to create history.")
    st.stop()

sess_df = pd.DataFrame(sessions)
st.markdown("### 📋 Session List")
st.dataframe(sess_df, use_container_width=True, hide_index=True)
st.divider()

# Risk trend line
if len(sess_df) > 1:
    st.markdown("### 📈 Risk Trend")
    trend = sess_df[["started_at","max_risk","critical_ct","high_ct"]].copy()
    trend = trend.sort_values("started_at")
    fig = px.line(trend, x="started_at",
                  y=["max_risk","critical_ct","high_ct"],
                  markers=True,
                  labels={"value":"Count/Score","variable":"Metric","started_at":"Scan Time"},
                  color_discrete_map={
                      "max_risk":   "#e74c3c",
                      "critical_ct":"#f1c40f",
                      "high_ct":    "#e67e22",
                  })
    fig.update_layout(
        paper_bgcolor="#12122a", plot_bgcolor="#1a1a3e",
        font=dict(color="#e2e8f0"), height=340,
        xaxis=dict(gridcolor="#1a1a3e"), yaxis=dict(gridcolor="#1a1a3e"),
    )
    st.plotly_chart(fig, use_container_width=True)
    st.divider()

# View / delete session
st.markdown("### 🔍 View Session Details")
sel_id = st.selectbox("Select session:",
                       options=[s["id"] for s in sessions],
                       format_func=lambda i: next(
                           f"#{i} - {s['started_at']} - {s['targets']}"
                           for s in sessions if s["id"] == i
                       ))

c1, c2 = st.columns([3,1])
with c1:
    if st.button("📂 Load Session", use_container_width=True):
        recs = get_session_records(sel_id)
        if recs:
            st.dataframe(pd.DataFrame(recs), use_container_width=True, hide_index=True, height=400)
        else:
            st.warning("Session has no records.")
with c2:
    if st.button("🗑️ Delete Session", use_container_width=True):
        delete_session(sel_id)
        st.success(f"Session #{sel_id} deleted.")
        st.rerun()

Overwriting dashboard/pages/5_History.py


In [13]:
%%writefile README.md
# 🛡️ NetGuard - Cyber Risk Assessment & Threat Intelligence Platform

NetGuard is a Python-based network security assessment tool that automates
vulnerability discovery, threat intelligence enrichment, and risk reporting.

---

## Prerequisites

| Tool | Version | Download |
|------|---------|----------|
| Python | 3.9+ | https://python.org |
| Nmap | 7.94+ | https://nmap.org/download.html |
| VirusTotal account | Free | https://www.virustotal.com/gui/join-us |
| Gmail account with 2FA | - | https://myaccount.google.com/security |

---

## Quick Start

### 1. Install dependencies

```bash
pip install -r requirements.txt
```

### 2. Configure credentials

Edit the `.env` file and fill in your values:

```env
VT_API_KEY=your_virustotal_api_key_here
GMAIL_SENDER=your_gmail@gmail.com
GMAIL_PASSWORD=xxxxxxxxxxxxxxxx
GMAIL_RECIPIENT=admin@yourdomain.com
SCAN_TARGETS=testphp.vulnweb.com,testasp.vulnweb.com
```

### 3. Gmail App Password setup

1. Enable 2-Step Verification: https://myaccount.google.com/security
2. Go to App Passwords: https://myaccount.google.com/apppasswords
3. Select **Mail** and generate a password
4. Copy the 16-character password (remove spaces) into `.env`

### 4. Run the dashboard

```bash
streamlit run dashboard/app.py
```

Open: http://localhost:8501

### 5. (Optional) Run from Jupyter Notebook

```bash
jupyter notebook main.ipynb
```

Run all cells in order to write all module files, then run the dashboard.

---

## Architecture

```
netguard/
├── modules/
│   ├── scanner.py       Nmap scanning + VirusTotal enrichment
│   ├── analyser.py      Risk scoring (CVSS-based)
│   ├── database.py      SQLite scan history
│   └── emailer.py       HTML email + PDF report
├── dashboard/
│   ├── app.py           Overview page (entry point)
│   └── pages/
│       ├── 2_Scan_Data.py
│       ├── 3_Charts.py
│       ├── 4_Threat_Intel.py
│       └── 5_History.py
├── main.ipynb
├── .env
├── .gitignore
├── requirements.txt
└── license.txt
```

---

## Recommended Test Targets

These are intentionally vulnerable or publicly accessible test targets:

| Target | Description |
|--------|-------------|
| `testphp.vulnweb.com` | Acunetix PHP test site - HTTP, FTP, MySQL exposed |
| `testasp.vulnweb.com` | Acunetix ASP test site - HTTP, SMTP open |
| `testaspnet.vulnweb.com` | Acunetix ASP.NET test site |
| `zero.webappsecurity.com` | HP Zero Bank - demo banking app |
| `pentest-ground.com` | Pentesting practice environment |
| `demo.testfire.net` | IBM Altoro Mutual - legacy demo bank |
| `demo.owasp-juice.shop` | OWASP Juice Shop - intentionally insecure app |
| `scanme.nmap.org` | Official Nmap scan-me target |

> ⚠️ **Only scan targets you own or have explicit written permission to scan.**
> Unauthorized scanning may violate computer misuse laws in your jurisdiction.

---

## Risk Scoring

Each finding is scored 1–10:

```
risk_score = min(10,  1  +  service_bonus  +  vt_malicious_count)
```

| Score | Severity | Action |
|-------|----------|--------|
| 9–10 | Critical | Remediate immediately |
| 7–8  | High     | Remediate within 24 hours |
| 4–6  | Medium   | Schedule remediation |
| 1–3  | Low      | Monitor |

---

## GitHub Repository Structure

Push the following to your GitHub repository:

```
netguard/
├── (all project files above)
├── docs/
│   ├── Project_Report.docx
│   ├── Agile_Documentation.docx
│   └── Presentation.pptx
├── assignments/          ← folder for previous assignments
│   └── (your previous assignment files)
└── license.txt
```

**Steps to push to GitHub:**
```bash
git init
git add .
git commit -m "Initial commit - NetGuard v1.0"
git remote add origin https://github.com/YOUR_USERNAME/netguard.git
git push -u origin main
```

---

## Troubleshooting

**Nmap not found**
```bash
# Windows (Chocolatey)
choco install nmap
# Linux
sudo apt-get install nmap
# macOS
brew install nmap
```

**Gmail authentication failed**
- Use an App Password, not your regular Gmail password
- Remove spaces from the App Password before pasting into `.env`
- Ensure 2FA is enabled first

**VT_API_KEY not working**
- Get a free key at https://www.virustotal.com/gui/my-apikey
- Free tier: 500 requests/day, 4 requests/minute

---

## License

MIT - see `license.txt`

Overwriting README.md


In [14]:
# ============================================================
# CELL - Quick validation test (no real scan)
# ============================================================
import importlib, sys, os, pandas as pd
sys.path.insert(0, '.')

import modules.scanner  as scanner_mod
import modules.emailer  as emailer_mod
import modules.database as db_mod
for m in [scanner_mod, emailer_mod, db_mod]:
    importlib.reload(m)

# Sample data matching pipeline output
sample = [
    {'ip':'192.168.1.10','hostname':'host-1','port':'22','service':'ssh',
     'product':'OpenSSH','version':'8.9','malicious_reports':0,
     'suspicious_count':0,'harmless_count':72,'community_score':10,
     'country':'IN','network':'192.168.1.0/24','categories':'',
     'risk_score':2,'severity':'Low','vulnerability':'SSH Exposed',
     'cve_ref':'CVE-2023-38408','cvss':5.0,'action':'Restrict SSH; use key auth.'},
    {'ip':'10.0.0.5','hostname':'target-2','port':'21','service':'ftp',
     'product':'vsftpd','version':'3.0.3','malicious_reports':3,
     'suspicious_count':1,'harmless_count':20,'community_score':-4,
     'country':'RU','network':'10.0.0.0/8','categories':'malware',
     'risk_score':7,'severity':'High','vulnerability':'Cleartext FTP',
     'cve_ref':'CVE-1999-0497','cvss':7.5,'action':'Disable FTP; use SFTP.'},
    {'ip':'10.0.0.5','hostname':'target-2','port':'23','service':'telnet',
     'product':'','version':'','malicious_reports':3,
     'suspicious_count':1,'harmless_count':20,'community_score':-4,
     'country':'RU','network':'10.0.0.0/8','categories':'malware',
     'risk_score':8,'severity':'Critical','vulnerability':'Cleartext Telnet',
     'cve_ref':'CVE-1999-0619','cvss':9.8,'action':'Disable Telnet immediately.'},
]

df = pd.DataFrame(sample)
print('Sample DataFrame:')
print(df[['ip','port','service','risk_score','severity']].to_string(index=False))

# Test PDF generation
os.makedirs('reports', exist_ok=True)
alert_df = df[df['severity'].isin(['High','Critical'])]
pdf_bytes = emailer_mod.build_pdf_report(alert_df, '2026-03-26 10:00:00', 8)
with open('reports/test_report.pdf', 'wb') as f:
    f.write(pdf_bytes)
print(f'\n✅ PDF: reports/test_report.pdf  ({len(pdf_bytes):,} bytes)')

# Test database
db_mod.init_db()
sid = db_mod.save_scan(['test-target'], sample)
print(f'✅ DB: session #{sid} saved')
print(f'✅ DB stats: {db_mod.get_db_stats()}')

print('\n🎉 All checks passed!')
print('▶  Run the dashboard: streamlit run dashboard/app.py')
print("Browse : http://localhost:8501")

Sample DataFrame:
          ip port service  risk_score severity
192.168.1.10   22     ssh           2      Low
    10.0.0.5   21     ftp           7     High
    10.0.0.5   23  telnet           8 Critical

✅ PDF: reports/test_report.pdf  (1,274 bytes)
✅ DB: session #22 saved
✅ DB stats: {'total_sessions': 16, 'total_records': 53, 'critical_total': 4, 'high_total': 8}

🎉 All checks passed!
▶  Run the dashboard: streamlit run dashboard/app.py
Browse : http://localhost:8501
